# Designing the Tingen City Map — End-to-End Walkthrough

A rough but complete tour of how the Tingen city map is built, from a blank prompt to a
walkable Godot scene whose **pathfinding routes around the buildings**. Written for Artemis
and Maxine ahead of our navigation chat (and for the rest of the lab).

The pipeline is five moves:

1. **Generate** a true top-down district map (`gpt-image-1`, flat reference + high fidelity).
2. **Relabel** the place names to *Lord of Mysteries* canon.
3. **Strip the buildings** off to get bare walkable ground (`map_bg.png`).
4. **Stamp** keyed building sprites back onto the ground.
5. **Drop colliders** = per-building footprints, so the engine's nav routes around them.

The deep reference is [`MAP_TUTORIAL.md`](MAP_TUTORIAL.md); the art rules are
[`STYLE_GUIDE.md`](STYLE_GUIDE.md). This notebook is the runnable narrative version.

## Tooling & attribution (please read)

Two image tools were used, and it matters which produced what:

- **Scripted API pipeline** — `generate_tingen_image2.py`, calling OpenAI's **`gpt-image-1`**
  Images API. This is the reproducible path; every call is logged to `manifest_image2.json`.
- **ChatGPT in-app image generator (GPT-Image, the "image 2" tool in the ChatGPT UI)** —
  used *interactively* by Mark for a few edits that were faster to iterate by hand than to
  script. **This is not captured in any script or manifest**, so it's credited explicitly here:
  - `my_assets/map_bg.png` (the bare-ground / building-strip in **Step 4**) was produced in the
    ChatGPT in-app image tool, not the API.
  - Several one-off `my_assets/` exports (e.g. the file literally named
    `ChatGPT Image Jun 18, 2026, 02_57_26 PM.png`) came from the same in-app tool.

> Same underlying model family as the API; the difference is *interactive chat* vs. *scripted
> call*. Where a step was done by hand in ChatGPT, the notebook says so inline.

## The one constraint that shapes everything: `gpt-image-1` has no seed

You cannot reproduce an image by fixing a random seed — there isn't one. The **only** levers
for consistency are:

1. the **reference image(s)** you pass, and
2. **`input_fidelity`** (`low` | `high`) — how tightly the output clings to the reference.

Two endpoints, chosen by whether you have a reference:

| Endpoint | When | Body |
|---|---|---|
| `POST /v1/images/edits` | you pass a reference image | **multipart**, `image[]=@ref.png` |
| `POST /v1/images/generations` | no reference (text only) | **JSON** |

Cost (approx): `low` ~$0.02 · `medium` ~$0.05 · `high` ~$0.25 per image. **Draft on `low`,
finalize on `high`.**

## 0. Setup

In [ ]:
# pip install requests pillow numpy   (scipy optional, speeds up the keying step)
import os, base64, requests
from pathlib import Path
from IPython.display import Image as ShowImage, display

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')  # or load from the .env the pipeline reads
ASSETS = Path('.')  # run this notebook from asset-gen/

def gen_image(prompt, ref=None, size='1536x1024', fidelity='high',
              quality='high', background='opaque', out='out.png'):
    """Minimal version of what generate_tingen_image2.py does under the hood.
    With a ref -> /images/edits (multipart). Without -> /images/generations (JSON)."""
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    if ref:
        files = {'image[]': (Path(ref).name, open(ref, 'rb'), 'image/png')}
        data = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size,
                'quality': quality, 'background': background,
                'output_format': 'png', 'input_fidelity': fidelity, 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/edits',
                          headers=headers, files=files, data=data, timeout=1800)
    else:
        body = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size,
                'quality': quality, 'background': background,
                'output_format': 'png', 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/generations',
                          headers={**headers, 'Content-Type': 'application/json'},
                          json=body, timeout=1800)
    r.raise_for_status()
    Path(out).write_bytes(base64.b64decode(r.json()['data'][0]['b64_json']))
    return out

# NOTE: latency is wildly variable (60s typical, but warehouse_interior once took ~24 min).
# In production, run generate_tingen_image2.py as a background job, not a foreground call.

## 1. Generate the base district map

**The hard problem:** a prompt asking for a "top-down map" *always* drifts to a ¾/oblique
angle. We tried 7 style variants; prose alone never holds a true 90° overhead.

**The fix:** feed a **flat-overhead reference** and set **`input_fidelity=high`** — high
fidelity locks the straight-down angle to the reference. (Trade-off: it also inherits the
reference's palette/contrast — that's the fight in Step 2.)

In [ ]:
# The flat-overhead reference that locks the 90 degree angle:
display(ShowImage(filename='ref/tingen_map.png', width=480))

In [ ]:
# Via the pipeline (preferred — resumable, logged):
#   python3 generate_tingen_image2.py \
#     --category backgrounds --treatment district_flat \
#     --only oldtown_core_flat3 --quality high
#
# What it does under the hood (equivalent direct call):
BG_DISTRICT_FLAT = (
    'a large highly detailed city district map seen from directly straight above at a '
    'TRUE flat 90-degree birds-eye angle, only rooftops and streets seen flat from above, '
    'NO perspective, NO tilt, NO isometric angle, NO building fronts'
)
# gen_image(BG_DISTRICT_FLAT, ref='ref/tingen_map.png', size='1536x1024',
#           fidelity='high', out='out_image2/backgrounds/oldtown_core_flat3.png')

## 2. Polish the prompt for contrast

The first flat map came out a **monochrome orange wash** — the high-fidelity reference plus a
global "golden daylight" instruction collapsed streets and roofs to one mid-orange value.

**The fix that worked** — force a light-vs-dark split *in the prompt*:
- **Streets:** pale, almost-white cool-grey paving — explicitly *not* golden, and *lighter than
  every roof*.
- **Roofs:** a vivid **patchwork** — terracotta next to charcoal-slate next to verdigris — so any
  two adjacent buildings differ in **both hue and brightness**.
- **Negatives:** `NOT a monochrome wash, NOT one hue, NOT one brightness, NOT hazy, NOT low-contrast`.

Keep the **reference + `input_fidelity=high` constant** so the *layout* stays put while you tune
only the palette wording. Iterate on `--quality low`, then one final `--quality high`.

## 3. Relabel place names to canon

The generated map has fictional/garbled labels. Relabel to canon (Iron Cross Market, Saint
Selena's Cathedral, the Docks, …) with an **image-to-image edit**: pass the map *itself* as the
reference at `input_fidelity=high` so art + layout are preserved, and change **only** the text.

⚠ **The text caveat:** AI image tools garble lettering (our first map literally produced
"GREENMARKET SQUARK"). Two safer routes, in order of preference:
1. **Generate the map label-free and add text by hand** (Preview / Photopea) — sharp, correct, free.
2. Label only ~5 districts via the API, then fix spelling manually.

Verify names against `../tingen_npc_roster.md` and the GDD §6.2 before baking them in.

In [ ]:
# Image-to-image relabel (same /images/edits call, map as its own reference):
RELABEL = (
    'Keep this exact map - same style, same layout, same top-down angle. '
    'Change ONLY the text labels to: Iron Cross Market, The Laughing Eel, '
    "Saint Selena's Cathedral, the Docks. Spell every label exactly as written. "
    'Make no other changes.'
)
# gen_image(RELABEL, ref='my_assets/map_v2.png', fidelity='high', out='my_assets/map_v3.png')

# The labeled overview map the game uses for look-at / fast-travel:
display(ShowImage(filename='my_assets/map_v3.png', width=520))

## 4. Strip the buildings → bare walkable ground

The walkable layer needs **streets only**, buildings removed, so building sprites can be stamped
on top in Step 5. The empty lots barely need detail — they'll be covered — they just need to read
as *streets*.

Two routes:

**A. API edit (scriptable).** Feed the clean map as the reference at `input_fidelity=high` (keeps
the exact street geometry) and prompt the removal.

**B. ChatGPT in-app image tool (what actually made the current `map_bg.png`).** The same edit —
upload the map, ask to remove the buildings — done interactively in the ChatGPT image UI. *This is
the in-app GPT-Image path credited at the top of the notebook; `my_assets/map_bg.png` came from
here, not the script.*

In [ ]:
# Route A, the scriptable edit:
STRIP = (
    'Remove ALL buildings, rooftops and labels. Keep ONLY the cobblestone streets, lanes, '
    'squares and the empty ground lots between them, in the exact same layout. Flat top-down. '
    'Empty paved/dirt lots where buildings were.'
)
# gen_image(STRIP, ref='my_assets/map_v3.png', size='1254x1254', fidelity='high',
#           out='my_assets/map_bg.png')

# The bare ground (made via Route B, ChatGPT in-app image tool):
display(ShowImage(filename='my_assets/map_bg.png', width=520))

## 5. Key out building sprites and stamp them on the ground

Each building is a self-contained iso compound (building + its grounds: e.g.
`St_Selena_Chapel_v2.png` = church + churchyard + fence). They usually arrive on a solid white
background, so we **key** them to transparent PNG with `key_building.py`, which does three things
in order, each fixing a different artifact:

1. **KEY** — mark *border-connected* near-white as background (so bright pixels *inside* the
   building are kept).
2. **ERODE** — shrink the alpha edge a few px to drop the anti-aliased fringe ring.
3. **BLEED** — fill transparent pixels' RGB with the nearest opaque colour, so the engine's LINEAR
   texture filtering never samples white (kills the white halo around a cutout).

In [ ]:
# Reusable for any building sprite on a white/pale background:
#   python3 key_building.py my_assets/St_Selena_Chapel_v2.png tingen/assets/props/chapel.png \
#       --white 222 --erode 2
#
# Then in Godot: map_bg.png as the Sprite2D ground (scale ~5, texture_filter = Linear),
# each keyed building stamped on its lot at the SAME iso angle and a fixed Klein-to-doorway
# scale, with the camera held at gameplay zoom (the iso-on-topdown mismatch only shows zoomed out).

display(ShowImage(filename='my_assets/St_Selena_Chapel_v2.png', width=320))

## 6. Drop colliders so pathfinding routes around the buildings

This is the step Chris asked about. Once buildings sit on the bare ground, the engine needs to know
**they are obstacles** — the ground is walkable, the building footprints are not — so NPC / agent
navigation routes *around* them.

### The authoring trick: read footprints in scene units, not image pixels
The generated image and the Godot scene are at different scales, so we overlay a **scene-coordinate
grid** on the image and read each footprint directly in world units. For the Klein room,
`image_px * 0.5833 = scene_unit` (a 1536×1024 image → ~896×597 scene units); `_scenegrid.py` draws
that grid and `_collidermap.py` draws the candidate boxes so you can eyeball them before committing.
The same method scales to the city map (its `Ground` polygon is 2400×1700 world units).

In [ ]:
# Core of _scenegrid.py / _collidermap.py: px<->scene conversion + footprint boxes.
from PIL import Image, ImageDraw

S = 0.5833            # image_px -> scene_unit (per-scene constant; measure once per map)
INV = 1.0 / S         # scene_unit -> image_px

def draw_footprints(img_path, boxes, out_path):
    """boxes: list of (cx, cy, w, h, label) in SCENE units. Draws each as a rectangle
    on the image so you can sanity-check footprints before exporting colliders."""
    im = Image.open(img_path).convert('RGB')
    d = ImageDraw.Draw(im, 'RGBA')
    for cx, cy, w, h, label in boxes:
        x0, y0 = (cx - w/2)*INV, (cy - h/2)*INV
        x1, y1 = (cx + w/2)*INV, (cy + h/2)*INV
        d.rectangle([x0, y0, x1, y1], outline=(0, 255, 255, 255), width=3)
        d.text((x0+3, y0+2), label, fill=(0, 255, 255, 255))
    im.save(out_path)
    return out_path

# Example (the real Klein-room furniture footprints from _collidermap.py):
room_boxes = [(78,150,155,70,'desk'), (390,167,204,265,'BED'), (685,85,174,114,'wardrobe')]
# draw_footprints('out_image2/klein_room/room_blood.png', room_boxes, '_check.png')

### From footprint boxes to Godot collision + navigation
Each footprint becomes a **`StaticBody2D` + `CollisionPolygon2D`** on the map. That's exactly what
`tingen/scenes/CityBlocks.tscn` already contains — a `Buildings` node with **53** such bodies on a
single walkable `Ground` polygon, e.g.:

```gdscript
[node name="Ground" type="Polygon2D" parent="."]
polygon = PackedVector2Array(0, 0, 2400, 0, 2400, 1700, 0, 1700)   # the walkable floor

[node name="B_r0_c0" type="StaticBody2D" parent="Buildings"]
[node name="CollisionPolygon2D" type="CollisionPolygon2D" parent="Buildings/B_r0_c0"]
polygon = PackedVector2Array(-76.8, -58.8, 76.8, -58.8, 76.8, 58.8, -76.8, 58.8)  # footprint
```

`City.tscn` adds a `Solids` body for the map edges + the cathedral, and `Area2D` door triggers
(`ChapelDoor`, `NeilHomeDoor`) for scene transitions.

**How this drives pathfinding:** the walkable `Ground` minus the union of building footprints **is**
the navigable region. Feed that to a `NavigationRegion2D` (or carve the footprints out as navigation
obstacles) and Godot's `NavigationServer2D` routes agents around the buildings automatically. The
`B_r{row}_c{col}` naming is from the generate-the-grid step — *generate → strip → colliders* — so the
footprints can be emitted programmatically rather than hand-placed.

### Why this is the bridge to Maxine and Artemis
Stripped down, the map is now exactly a **navigation problem on a grid/graph**:

- a **walkable field** (the bare ground), and
- a set of **obstacle footprints** (the building colliders).

- **Maxine (maze navigation):** the footprint grid *is* a maze — same routing/obstacle-avoidance
  problem, just sourced from a generated city instead of a synthetic maze.
- **Artemis (VLM navigation / AI2 synthetic maps):** the `map_v3` (labeled, semantic) + `map_bg`
  (bare walkable) pair is a natural input for VLM-driven navigation, and the generate→strip→collider
  recipe is a way to mint synthetic-map + ground-truth-navmesh pairs.

## Run order & file map

| Stage | Command / file | Output |
|---|---|---|
| 1–2 generate + polish | `generate_tingen_image2.py --category backgrounds --treatment district_flat` | `out_image2/backgrounds/*` |
| 3 relabel | image-to-image edit (prefer manual text) | `my_assets/map_v3.png` |
| 4 strip buildings | API edit **or ChatGPT in-app tool** | `my_assets/map_bg.png` |
| 5 key sprites | `key_building.py SRC OUT --white 222 --erode 2` | `tingen/assets/props/*.png` |
| 6 colliders | `_scenegrid.py` / `_collidermap.py` → `CityBlocks.tscn` | `Buildings/*` StaticBody2D |

### TL;DR
flat ref + `input_fidelity=high` to lock the 90° angle → tune palette (pale streets vs patchwork
roofs) for contrast → relabel to canon (prefer manual text) → strip the buildings to get `map_bg`
→ key building sprites and stamp them at a locked iso angle → author per-building footprint colliders
(`StaticBody2D` + `CollisionPolygon2D`) so the walkable ground minus footprints is the navigable region
and pathfinding routes around the buildings.

*Image credits: `gpt-image-1` API via `generate_tingen_image2.py`, except `map_bg.png` and assorted
`my_assets/` exports, which were made with the ChatGPT in-app image generator (GPT-Image / "image 2").*